# Krionis Regulatory Wire Walkthrough

This notebook walks through the dedicated regulatory wire in Krionis, including the new runtime-selection and signoff helper APIs:

1. Create a regulation-only data pool
2. Rebuild the pool index for the selected embedding model
3. Start the built-in `regulatory` agent with an explicit runtime profile and Hugging Face model selections
4. Run a compliance assessment against the pool
5. Fetch signoff-ready approval instructions
6. Review and approve the flagged assessment

Before running the notebook, start the local Krionis API and set the review API key.


In [ ]:
import json
import os

import requests

BASE_URL = os.environ.get("KRIONIS_BASE_URL", "http://127.0.0.1:8000")
REVIEW_API_KEY = os.environ.get("KRIONIS_REVIEW_API_KEY", "local-review-key")

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})

BASE_URL

## 1. Create a regulation-only pool

Point Krionis at a folder that contains only regulatory material.

In [ ]:
pool_payload = {
    "name": "EURegulations",
    "docs_dir": "data/manuals/regulations",
    "framework": "EU GMP",
    "focus": "Validation"
}

pool_response = session.post(f"{BASE_URL}/compliance/pools", data=json.dumps(pool_payload))
pool_response.raise_for_status()
pool_response.json()

## 2. Rebuild the pool index

After adding regulation documents to the pool directory, rebuild the retrieval cache.

In [ ]:
rebuild_response = session.post(f"{BASE_URL}/compliance/pools/EURegulations/rebuild")
rebuild_response.raise_for_status()
rebuild_response.json()

## 3. Start the built-in regulatory agent with a runtime profile


In [ ]:
agent_payload = {
    "system": "EURegulations",
    "agent_type": "regulatory",
    "name_prefix": "regulatory",
    "runtime_profile": "balanced-search",
    "inference_model": "qwen-1.5b-instruct",
    "embedding_model": "bge-small-en-v1.5"
}

agent_response = session.post(f"{BASE_URL}/platform/agents/start", data=json.dumps(agent_payload))
agent_response.raise_for_status()
agent_info = agent_response.json()
agent_info


## 4. Submit a regulated document for compliance assessment

In [ ]:
assessment_payload = {
    "document_name": "Validation SOP Draft",
    "document_text": "The procedure describes execution steps but does not include validation evidence or approval signatures.",
    "regulation_system": "EURegulations",
    "framework": "EU GMP Annex 11",
    "focus": "Validation",
    "runtime_profile": "balanced-search",
    "inference_model": "qwen-1.5b-instruct",
    "embedding_model": "bge-small-en-v1.5"
}

assessment_response = session.post(f"{BASE_URL}/compliance/assess", data=json.dumps(assessment_payload))
assessment_response.raise_for_status()
assessment = assessment_response.json()
assessment


## 5. Fetch signoff instructions and approve the assessment

Compliance outputs commonly route into HITL review because they are regulated outputs. The signoff helper shows the exact approval and rejection API contract before you post the final decision.


In [ ]:
review_id = assessment["review_id"]

signoff_response = session.get(f"{BASE_URL}/review/{review_id}/signoff", headers={"x-api-key": REVIEW_API_KEY})
signoff_response.raise_for_status()
signoff = signoff_response.json()
signoff

review_headers = {
    "Content-Type": "application/json",
    "x-api-key": REVIEW_API_KEY,
    "x-reviewer-id": "qa-lead-1"
}

approval_payload = {
    "final_response": "Approved compliance assessment after QA review.",
    "reviewer_notes": "Validated against current procedural controls."
}

approval_response = requests.post(
    f"{BASE_URL}/review/{review_id}/approve",
    headers=review_headers,
    data=json.dumps(approval_payload)
)
approval_response.raise_for_status()
approval_response.json()


## 6. Inspect stored compliance assessments

In [ ]:
stored_response = session.get(f"{BASE_URL}/compliance/assessments")
stored_response.raise_for_status()
stored_response.json()